## 2.1 Imputing, Encoding and First Pipeline

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from features import *

data = pd.read_csv('train.csv')
child_feature(data)
family_size_feature(data)

In [14]:
X = data.drop(columns=['Survived', 'Ticket', 'PassengerId', 'Name', 'Cabin'])
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Child,FamilySize
0,3,male,22.0,1,0,7.2500,S,0,2
1,1,female,38.0,1,0,71.2833,C,0,2
2,3,female,26.0,0,0,7.9250,S,0,1
3,1,female,35.0,1,0,53.1000,S,0,2
4,3,male,35.0,0,0,8.0500,S,0,1


### Categorical and numerical features

In [15]:
cat_features = list(X.drop(columns=X.select_dtypes(include=['int', 'float'])))
num_features = list(X.drop(columns=X.select_dtypes(include=['str'])))

### Numerical and categorical pipeline + Imputer

In [16]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

### ColumnTransfer

In [17]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

### Final Pipeline

In [18]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=1.0, max_iter=1000))
])

final_pipeline.fit(X_train, y_train)
prediction = final_pipeline.predict(X_test)

### Confusion Matrix, predict_proba, precision, recall

In [19]:
conf_matrix = confusion_matrix(y_test, prediction)
print(conf_matrix)

proba_survived = final_pipeline.predict_proba(X_test)[:, 1]
new_labels = (proba_survived >= 0.7).astype(int)

new_conf_matrix = confusion_matrix(y_test, new_labels)
print(new_conf_matrix)

[[89 16]
 [21 53]]
[[101   4]
 [ 35  39]]


### Important to mention:
 - 1. We have to increase recall because it is better to indentify person as dead and he is not than identify person as alive and he is dead